In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[int(y_true[i])][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [5]:
from sklearn.metrics import classification_report
ths = [20, 20.5, 21, 21.5, 22, 23, 24, 25]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,0,0,0,0,0,202228
1,0,0,0,0,0,48,128488
2,0,0,64293,0,0,0,18012
3,0,0,0,42877,0,0,18073
4,0,0,0,0,40287,0,5503
5,0,0,0,0,4,24,119
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.6726742262806853 recall:1.0 precision:0.5430062053095539 f1:0.7038289326913204
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.54      1.00      0.70    202228
           1       0.00      0.00      0.00    128536
           2       1.00      0.78      0.88     82305
           3       1.00      0.70      0.83     60950
           4       1.00      0.88      0.94     45790
           5       0.33      0.16      0.22       147

    accuracy                           0.67    519956
   macro avg       0.65      0.59      0.59    519956
weighted avg       0.57      0.67      0.59    519956



d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

[0, 1, 2, 3, 4, 5, -1]
th = 20.5 hidden = 0


...

d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,53714,0,0,0,0,0,148514
1,0,0,0,0,0,0,128536
2,0,0,41831,0,0,0,40474
3,0,0,0,36248,0,0,24702
4,0,0,0,0,19971,0,25819
5,0,0,0,0,0,0,147
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.29216125979890606 recall:1.0 precision:0.0003992482183208761 f1:0.0007981777655909366
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      1.00      0.00       147
           0       1.00      0.27      0.42    202228
           1       0.00      0.00      0.00    128536
           2       1.00      0.51      0.67     82305
           3       1.00      0.59      0.75     60950
           4       1.00      0.44      0.61     45790

    accuracy                           0.29    519956
   macro avg       0.67      0.47      0.41    519956
weighted avg       0.75      0.29      0.41    519956



d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 20: 0.40235852191048715
Média f1 absolute th 20.5: 0.3915279531877827
Média f1 absolute th 21: 0.38229031274745817
Média f1 absolute th 21.5: 0.3714439488079409
Média f1 absolute th 22: 0.35549788922083503
Média f1 absolute th 23: 0.32321571012758876
Média f1 absolute th 24: 0.29773936703122544
Média f1 absolute th 25: 0.28865229588042246
